# Predict - Recalibrate

- Takes a dummy ecg and outcome file and outputs a probability matrix
- One example per type of MI (STEMI, OMI, NSTEMI_type1, NSTEMI_type2)



In [1]:
# Import packages
import contextlib

import joblib
import lightning as L
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import torch
from torch.utils.data import DataLoader

from hparams import hparams_dict
from h5dataset import H5ECGOnly, outcome_labels
from model import LitResNet
from recalibrate import MergedIsotonicModel


/home/ubuntu/miniconda3/envs/malbec_ecg/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/miniconda3/envs/malbec_ecg/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `<class 'pathlib.Path'>` but got `<class 'NoneType'>` with value `'None'` - serialized value may not be as expected.)
  PydanticSerializationUnexpectedValue(Expected `<class 'pathlib.Path'>` but got `<class 'NoneType'>` with value `'None'` - serialized value may not be as expected.)
  return self.__pydantic_serializer__.to_python(


In [2]:
path= '../data/ecgs_dummy.h5'
target_file='../data/outcome_dummy.csv'

# Initialise ECG and outcome dataset
dataset_dummy = H5ECGOnly(path, 
                          target_file, 
                          n_classes=hparams_dict["n_classes"],
                          mean_=-0.5810575960776053,
                          std_=328.8951471302853)

H5ECG samples 5


In [3]:
# Create dataloader
loader = DataLoader(dataset_dummy)

# Load model
checkpoint_path = "../models/finetuned_18d5a8abc508439a82786289b8f67ccc.ckpt"


# Initialise model
model=LitResNet.load_from_checkpoint(checkpoint_path,hparams=hparams_dict)
with contextlib.redirect_stdout(None):
    model.eval();

trainer = L.Trainer(
    accelerator="gpu",  
    devices=1,          
    strategy="auto"
   #precision="32-true"
)
batch_logits = trainer.predict(model, dataloaders=loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/ubuntu/miniconda3/envs/malbec_ecg/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA A30') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_

Predicting DataLoader 0: 100%|██████████| 5/5 [00:00<00:00, 15.40it/s]


In [4]:
# Concatenates batch to single tensor
all_logits = torch.cat(batch_logits)

# Applies sigmoid to the whole tensor
probabilities_tensor = torch.sigmoid(all_logits)

# Convert back to dataframe
probabilities = pd.DataFrame(probabilities_tensor.cpu().numpy())
labels=["Prob_"+ l for l in outcome_labels[0:hparams_dict["n_classes"]]]
probabilities.rename(columns={old: new for old, new in zip(probabilities.columns[:6], labels)}, inplace=True)
probabilities

,Prob_stemi_hd_v2,Prob_nstemi_hd_v2,Prob_nstemi_type1_hd_v2,Prob_nstemi_type2_hd_v2,Prob_other_hd_v2,Prob_non_hd_v2
0,0.000003,0.000466,0.000370,0.000040,0.020077,0.994470
1,0.999984,0.005722,0.004430,0.000048,0.003961,0.000075
2,0.000963,0.309991,0.034239,0.279704,0.811221,0.023607
3,0.080542,0.678837,0.650932,0.044878,0.094662,0.108943
4,0.961268,0.116199,0.111246,0.003635,0.032405,0.015793


In [5]:
# Merge with ECG hashes
probabilities.reset_index(drop=True, inplace=True)
hash_=dataset_dummy.hash
hash_.reset_index(drop=True, inplace=True)
probabilities_hash_ = pd.concat([probabilities, hash_], axis=1).drop_duplicates(subset='hash')
probabilities_hash_

,Prob_stemi_hd_v2,Prob_nstemi_hd_v2,Prob_nstemi_type1_hd_v2,Prob_nstemi_type2_hd_v2,Prob_other_hd_v2,Prob_non_hd_v2,hash,index_o,index_e,group
0,0.000003,0.000466,0.000370,0.000040,0.020077,0.994470,non,3,0,000000
1,0.999984,0.005722,0.004430,0.000048,0.003961,0.000075,stemi,1,1,100000
2,0.000963,0.309991,0.034239,0.279704,0.811221,0.023607,nstemi_type2,2,2,010100
3,0.080542,0.678837,0.650932,0.044878,0.094662,0.108943,nstemi_type1,0,3,011000
4,0.961268,0.116199,0.111246,0.003635,0.032405,0.015793,omi,4,4,100000


In [6]:
probabilities_hash_["hash"] = probabilities_hash_["hash"].astype(str).str.strip()

# Read target file and left join target file with probabilities_hash_ 
outcome_dummy = pd.read_csv(target_file)
outcome_dummy["ECSE_BK_hash"] = outcome_dummy["ECSE_BK_hash"].astype(str).str.strip()

merged_df = outcome_dummy.merge(probabilities_hash_, left_on="ECSE_BK_hash", right_on="hash", how="inner")
merged_df

,ECSE_BK_hash,stemi_hd_v2,nstemi_hd_v2,nstemi_type1_hd_v2,nstemi_type2_hd_v2,other_hd_v2,non_hd_v2,Prob_stemi_hd_v2,Prob_nstemi_hd_v2,Prob_nstemi_type1_hd_v2,Prob_nstemi_type2_hd_v2,Prob_other_hd_v2,Prob_non_hd_v2,hash,index_o,index_e,group
0,nstemi_type1,0,1,1,0,0,0,0.080542,0.678837,0.650932,0.044878,0.094662,0.108943,nstemi_type1,0,3,011000
1,stemi,1,0,0,0,0,0,0.999984,0.005722,0.004430,0.000048,0.003961,0.000075,stemi,1,1,100000
2,nstemi_type2,0,1,0,1,0,0,0.000963,0.309991,0.034239,0.279704,0.811221,0.023607,nstemi_type2,2,2,010100
3,non,0,0,0,0,0,0,0.000003,0.000466,0.000370,0.000040,0.020077,0.994470,non,3,0,000000
4,omi,1,0,0,0,0,0,0.961268,0.116199,0.111246,0.003635,0.032405,0.015793,omi,4,4,100000


## Perform Recalibration

In [7]:
def predict_isotonic(X=None, model_filename: str="") -> tuple[pd.DataFrame, float]:
    """
    Make predictions using a saved isotonic regression model.

    Parameters:
    X (array-like): Probabilities to be recalibrated (1D array).
    model_filename (str): Base filename for model (without prefixes/extensions).
    String from array ['stemi_hd', 'nstemi_hd', 'nstemi_type1', 'nstemi_type2']

    Returns:
    tuple: (recalibrated probabilities])
    """
    
    # Ensure X is 1D array
    X = np.asarray(X).ravel()
    
    
    # Reconstruct filename matching calibration function's naming convention
    if model_filename != "":
        model_filename = f"../models/model_{model_filename}.pkl"
    else:
        model_filename = "../models/model_isotonic.pkl"

    # Load model and predict
    calibration_model = joblib.load(model_filename)
    recalibrated_probs = calibration_model.predict(X)

    return recalibrated_probs


def recalibrate_pred(df: pd.DataFrame, model_filename: str) -> tuple[pd.DataFrame, float]:
    """
    Recalibrates probabilties using isotonic regression
    
    df: uncalibrated merged dataframe
    model_filename (str): Base filename for model (without prefixes/extensions).
    String from array ['stemi_hd', 'nstemi_hd', 'nstemi_type1_hd', 'nstemi_type2_hd']
    """
    
    if model_filename in ["nstemi_hd", "nstemi_type1_hd"]:
        recalibrated_probs = predict_isotonic(df[["Prob_" + model_filename + "_v2","Prob_stemi_hd_v2"]].max(axis=1),                                                    
                                                            model_filename=model_filename)
    else:
        recalibrated_probs = predict_isotonic(df[["Prob_" + model_filename + "_v2"]],                                                    
                                                            model_filename=model_filename)

    df["Prob_" + model_filename + "_v2"] = recalibrated_probs
    return df

In [8]:
valid_MIs = ['stemi_hd']

for model_name in valid_MIs:
    merged_df = recalibrate_pred(merged_df, model_name)

print("STEMI probabilities recalibrated")
merged_df


STEMI probabilities recalibrated


,ECSE_BK_hash,stemi_hd_v2,nstemi_hd_v2,nstemi_type1_hd_v2,nstemi_type2_hd_v2,other_hd_v2,non_hd_v2,Prob_stemi_hd_v2,Prob_nstemi_hd_v2,Prob_nstemi_type1_hd_v2,Prob_nstemi_type2_hd_v2,Prob_other_hd_v2,Prob_non_hd_v2,hash,index_o,index_e,group
0,nstemi_type1,0,1,1,0,0,0,0.102147,0.678837,0.650932,0.044878,0.094662,0.108943,nstemi_type1,0,3,011000
1,stemi,1,0,0,0,0,0,1.000000,0.005722,0.004430,0.000048,0.003961,0.000075,stemi,1,1,100000
2,nstemi_type2,0,1,0,1,0,0,0.002333,0.309991,0.034239,0.279704,0.811221,0.023607,nstemi_type2,2,2,010100
3,non,0,0,0,0,0,0,0.000107,0.000466,0.000370,0.000040,0.020077,0.994470,non,3,0,000000
4,omi,1,0,0,0,0,0,0.910085,0.116199,0.111246,0.003635,0.032405,0.015793,omi,4,4,100000


In [9]:
valid_MIs = ['nstemi_hd', 'nstemi_type1_hd', 'nstemi_type2_hd']

#exclude stemis
merged_df = merged_df[merged_df["stemi_hd_v2"] == 0].copy() #Consistent with clinical practice, performance on the NSTEMI subtype was assessed after exclusion of STEMI patients. 
for model_name in valid_MIs:
    merged_df = recalibrate_pred(merged_df, model_name)

print("NSTEMI probabilities recalibrated")
merged_df #exclude stemis 

NSTEMI probabilities recalibrated


,ECSE_BK_hash,stemi_hd_v2,nstemi_hd_v2,nstemi_type1_hd_v2,nstemi_type2_hd_v2,other_hd_v2,non_hd_v2,Prob_stemi_hd_v2,Prob_nstemi_hd_v2,Prob_nstemi_type1_hd_v2,Prob_nstemi_type2_hd_v2,Prob_other_hd_v2,Prob_non_hd_v2,hash,index_o,index_e,group
0,nstemi_type1,0,1,1,0,0,0,0.102147,0.766175,0.710313,0.093554,0.094662,0.108943,nstemi_type1,0,3,011000
2,nstemi_type2,0,1,0,1,0,0,0.002333,0.442928,0.037603,0.450269,0.811221,0.023607,nstemi_type2,2,2,010100
3,non,0,0,0,0,0,0,0.000107,0.000789,0.000615,0.000098,0.020077,0.994470,non,3,0,000000
